In [ ]:
import os, sys, argparse, builtins, warnings, subprocess, time
BASE = "/Users/marcus/Documents/GitHub/Ai-plays-SubwaySurfers/alpha/arrow_save_to_transcend.py"
subprocess.run([sys.executable, BASE, "shutdown"], check=False)

Traceback (most recent call last):
  File "/Users/marcus/Documents/GitHub/Ai-plays-SubwaySurfers/alpha/arrow_save_to_transcend.py", line 10, in <module>
    from pynput import keyboard
ModuleNotFoundError: No module named 'pynput'


CompletedProcess(args=['/opt/anaconda3/envs/llms/bin/python', '/Users/marcus/Documents/GitHub/Ai-plays-SubwaySurfers/alpha/arrow_save_to_transcend.py', 'shutdown'], returncode=1)

: 

In [1]:
# one-cell, drop-in snippet to mimic:
#   _tap('left'); _tap('left'); lane = 0
# but also record precise timings

import time

# Toggle this to send real keypresses. Keep False to use a harmless mock.
USE_REAL_KEYPRESSES = False

if USE_REAL_KEYPRESSES:
    import pyautogui
    press_fn = pyautogui.press
else:
    # Minimal mock that mimics pyautogui.press without doing anything
    class TapMock:
        def __init__(self):
            self.calls = []
        def __call__(self, key):
            # record when we'd have pressed the key
            self.calls.append((key, time.perf_counter_ns()))
    tap_mock = TapMock()
    press_fn = tap_mock

def _tap(k: str, delay_s: float = 0.045):
    """
    Mimics your original _tap: try pyautogui.press(k); sleep(0.045).
    Returns (t_start, t_after_press, t_after_sleep) in ns for metrics.
    """
    t_start = time.perf_counter_ns()
    try:
        press_fn(k)
    except Exception:
        pass
    t_after_press = time.perf_counter_ns()
    time.sleep(delay_s)
    t_after_sleep = time.perf_counter_ns()
    return t_start, t_after_press, t_after_sleep

# ---- run the exact sequence and collect timings ----
lane = 1
t0, t1, t2 = _tap('left', 0.045)
t3, t4, t5 = _tap('left', 0.045)
lane = 0

# ---- compute metrics (ms) ----
ns2ms = lambda ns: ns / 1e6
metrics = {
    "requested_delay_ms": 0.045 * 1000.0,
    "press1_call_ms":     ns2ms(t1 - t0),
    "press1_sleep_ms":    ns2ms(t2 - t1),
    "press2_call_ms":     ns2ms(t4 - t3),
    "press2_sleep_ms":    ns2ms(t5 - t4),
    "start_to_start_ms":  ns2ms(t3 - t0),   # gap between the two tap starts
    "total_double_ms":    ns2ms(t5 - t0),   # full sequence duration
    "lane_after":         lane,
    "using_mock":         not USE_REAL_KEYPRESSES,
}

# ---- pretty print ----
print("Double-tap timing metrics:")
for k, v in metrics.items():
    print(f"  {k:>18}: {v:.3f}" if isinstance(v, float) else f"  {k:>18}: {v}")

if not USE_REAL_KEYPRESSES:
    print("\nMock press call timestamps recorded:", len(tap_mock.calls))
    for i, (key, ts) in enumerate(tap_mock.calls, 1):
        print(f"  call{i}: key={key!r} at {ts} ns")


Double-tap timing metrics:
  requested_delay_ms: 45.000
      press1_call_ms: 0.002
     press1_sleep_ms: 50.023
      press2_call_ms: 0.002
     press2_sleep_ms: 48.550
   start_to_start_ms: 50.113
     total_double_ms: 98.665
          lane_after: 0
          using_mock: True

Mock press call timestamps recorded: 2
  call1: key='left' at 1209986555270583 ns
  call2: key='left' at 1209986605383208 ns
